# VSP Agents: Sub-30ms Clinical Decision Engine 🏥⚡

**Author:** VSP Agents (Vishnu Suresh Perumbavoor)  
**Hardware:** NVIDIA A100-SXM4 (80GB VRAM)  
**Benchmark:** Microsoft BLURB (PubMedQA)  

### 📊 Official Benchmark Performance on 500-Sample Test Set:
* **Accuracy:** **71.4%** (**+59.0 points net gain** over baseline, +475% relative)
* **YES Class F1:** **78.6%**
* **NO Class F1:** **70.4%**
* **Inference Latency:** **<30ms** per verdict (non-autoregressive)
* **Hugging Face Model:** [vishnusureshperumbavoor/vspagent-pubmedqa-decision-a100](https://huggingface.co/vishnusureshperumbavoor/vspagent-pubmedqa-decision-a100)

In [ ]:
# Install transformers
!pip install -q transformers torch

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_id = "vishnusureshperumbavoor/vspagent-pubmedqa-decision-a100"
print(f"Loading {model_id} from Hugging Face Hub...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id)
model.eval()
print("✔ VSP Agents PubMedQA model loaded successfully!")

In [ ]:
# Live clinical trial reasoning test
question = "Do GLP-1 receptor agonists reduce cardiovascular events in type 2 diabetes?"
abstract = (
    "Large randomized cardiovascular outcome trials have conclusively demonstrated that "
    "GLP-1 receptor agonists significantly reduce major adverse cardiovascular events (MACE) "
    "and all-cause mortality in patients with type 2 diabetes."
)

input_text = f"Question: {question}\nAbstract: {abstract}\nDecision query: {question}"
inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512)

with torch.no_grad():
    logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0]

labels = model.config.id2label
print("=== VSP AGENTS CLINICAL VERDICT ===")
for i, p in enumerate(probs):
    pct = round(float(p) * 100, 1)
    bar = "█" * int(float(p) * 20)
    print(f"{labels[i].upper():<6} {pct:>5}% {bar}")